# Phase 0 — Colab setup, smoke test and benchmark

Runs GraphCast_small on a Colab GPU from this project's repository.

**Before running:** Runtime → Change runtime type → **T4 GPU** (free tier is fine).

Order: 1 check runtime → 2 get the code → 3 install → 4 storage → 5 download model → 6 offline test → 7 smoke test → 8 benchmark.

## 1. Check the runtime (Python ≥ 3.12 and a GPU)

In [ ]:
import sys, subprocess
print("Python", sys.version.split()[0])
assert sys.version_info >= (3, 12), "The GraphCast library needs Python >= 3.12"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "No GPU - switch the runtime type")

## 2. Get the code

Set `REPO_URL` to your GitHub repository. For a **private** repo, create a fine-grained
GitHub token (read-only, this repo only), store it in Colab's 🔑 *Secrets* panel as
`GITHUB_TOKEN`, and leave `PRIVATE = True`.

In [ ]:
REPO_URL = "https://github.com/<your-username>/graphcast-extremes"   # <- edit
PRIVATE = True

import os
url = REPO_URL
if PRIVATE:
    from google.colab import userdata
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
if not os.path.exists("/content/graphcast-extremes"):
    subprocess.run(["git", "clone", "-q", url, "/content/graphcast-extremes"], check=True)
else:
    subprocess.run(["git", "-C", "/content/graphcast-extremes", "pull", "-q"], check=True)
%cd /content/graphcast-extremes
!git log -1 --format="%h %s"

## 3. Install (≈3–5 min)

Installs the pinned requirements and this project. `requirements.txt` pins JAX 0.7.2
(the newer 0.12 build gave all-NaN forecasts on the Mac CPU); on Colab the next cell installs
the GPU (CUDA) build of the same version.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
!pip install -q "jax[cuda12]==0.7.2"
import jax
print(jax.__version__, jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU visible: Runtime -> Restart session, then rerun from step 2"

## 4. Storage

* `EXTREMES_DATA` → Colab's local disk (fast, wiped when the session ends; the model re-downloads in seconds).
* `EXTREMES_RESULTS` → your Google Drive, so results survive the session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["EXTREMES_DATA"] = "/content/data"
os.environ["EXTREMES_RESULTS"] = "/content/drive/MyDrive/graphcast-extremes/results"
os.makedirs(os.environ["EXTREMES_RESULTS"], exist_ok=True)
print(os.environ["EXTREMES_DATA"], "|", os.environ["EXTREMES_RESULTS"])

## 5. Download GraphCast_small, normalisation stats and the 1° example data

In [ ]:
!python scripts/00_download_assets.py

## 6. Offline test (random tiny model on synthetic data)

In [ ]:
!python -m pytest -q tests

## 7. Smoke test — must print PASSED

In [ ]:
!python scripts/01_smoke_test.py
from IPython.display import Image, display
display(Image(os.environ["EXTREMES_RESULTS"] + "/phase0/smoke_test.png"))

## 8. Timing benchmark

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().split()[-1].lower()
!python scripts/02_benchmark.py --label colab-{gpu}